# Exercise 06 — Working with Pre-trained Language Models

In the previous notebook you pretrained a 0.8 M parameter model for a few minutes on 1 MB of Shakespeare. The models in this notebook were made the same way — a Transformer, self-supervised training on raw text — but with a hundred times more parameters and many gigabytes of text. You will not train anything here: you **download** them with the Hugging Face [`transformers`](https://huggingface.co/docs/transformers/index) library and find out what pretraining has taught them.

There are three families of pre-trained Transformers. They differ in *which part* of the Transformer they keep and in their **pretraining objective**, and that decides what they are good at:

| family | model in this notebook | pretraining objective | each position sees | typically used for |
|---|---|---|---|---|
| encoder-only | **BERT** | *masked language modelling*: predict hidden tokens | the whole input, in both directions | understanding: classification, tagging, sentence and word vectors |
| decoder-only | **GPT-2** | *causal language modelling*: predict the next token | only the tokens to its left | generating text |
| encoder–decoder | **T5** | *span corruption*: write out the spans that were removed from the input | encoder: all of the input; decoder: its own output so far | text-to-text tasks: translation, summarisation, … |

## What you will do
1. **BERT** — fill in blanks, see why context on *both* sides matters, find out what else the model has picked up, and compare contextual word vectors.
2. **GPT-2** — compare it with your mini GPT, look at next-token distributions, measure perplexity, and sample text.
3. **T5** — span corruption, and one model for many tasks.
4. **Multiple-choice questions** — to check your understanding.

## How to work through it
Same as before: run the cells **in order**, fill in every `# TODO`, and make sure each **✅ Check** cell passes before moving on. Tasks that say *Your answer here* want a short written answer.

> **Heads up — this notebook downloads three models**, about 1.2 GB in total: BERT (~440 MB), GPT-2 (~550 MB) and T5-small (~240 MB). Each is downloaded the first time it is loaded and cached afterwards, so make sure you have a working internet connection. Everything runs on a CPU; nothing is trained.

In [ ]:
import math

import torch
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    pipeline,
)

torch.manual_seed(0)

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")


def show(predictions):
    # Print a list of (token, probability) pairs
    for token, p in predictions:
        print(f"    {token!r:<16} {p:.3f}")

## 1. BERT: an encoder that reads in both directions

[BERT](https://arxiv.org/abs/1810.04805) (Devlin et al., 2019) is a stack of 12 Transformer **encoder** layers: self-attention *without* a causal mask, so every token sees the whole sentence. It cannot be pretrained to predict the next token — it would simply look at it. Instead, 15% of the input tokens are hidden behind a `[MASK]` token, and the model has to reconstruct them: **masked language modelling** (MLM).

`AutoModelForMaskedLM` loads the encoder together with the MLM head, which turns every output vector into logits over the vocabulary — just like the `lm_head` of your GPT. (You may see a message that some weights were not used: those belong to BERT's second pretraining task, next sentence prediction, which we do not need.)

In [ ]:
BERT_NAME = "google-bert/bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
bert = AutoModelForMaskedLM.from_pretrained(BERT_NAME).to(device).eval()

print(f"{sum(p.numel() for p in bert.parameters()):,} parameters, a vocabulary of {bert_tokenizer.vocab_size:,} tokens")

BERT's tokenizer lowercases the text (this is the *uncased* model), splits it into **WordPiece** subword tokens, and adds two special tokens: `[CLS]` at the start (its output vector is used for classification) and `[SEP]` at the end (it separates two sentences in sentence-pair tasks).

In [ ]:
inputs = bert_tokenizer("The capital of France is [MASK].", return_tensors="pt")
print(inputs["input_ids"])
print(bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

### Filling in the blank

**1.1 Complete `mask_distribution`**, which returns BERT's probability distribution over the vocabulary at the `[MASK]` position. `fill_mask` uses it to list the most likely tokens.

Hints:
- `bert(**inputs)` returns an object with the attribute `.logits`, of shape `(1, seq_len, vocab_size)`.
- `(ids == bert_tokenizer.mask_token_id).nonzero()` finds the position of the mask.

In [ ]:
def mask_distribution(text):
    # BERT's probability for every token of the vocabulary at the (single) [MASK] in text, shape (vocab_size,)
    inputs = bert_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = bert(**inputs).logits
    mask_index = (inputs["input_ids"][0] == bert_tokenizer.mask_token_id).nonzero().item()
    return logits[0, mask_index].softmax(dim=-1)


def fill_mask(text, k=5):
    # The k most likely tokens for the [MASK], as a list of (token, probability)
    top = torch.topk(mask_distribution(text), k)
    return list(zip(bert_tokenizer.convert_ids_to_tokens(top.indices.tolist()), top.values.tolist()))


show(fill_mask("The capital of France is [MASK]."))

In [ ]:
# ✅ Check your mask_distribution
probs = mask_distribution("The capital of France is [MASK].")
assert probs.shape == (bert_tokenizer.vocab_size,), f"expected one probability per token of the vocabulary, got shape {tuple(probs.shape)}"
assert torch.isclose(probs.sum(), torch.tensor(1.0, device=probs.device), atol=1e-4), "the probabilities should sum to 1"
assert fill_mask("The capital of France is [MASK].")[0][0] == "paris", "BERT should know the capital of France"
print("Looks good ✅")

The Hugging Face **pipelines** wrap tokenization, the model and the post-processing into a single call. Your `fill_mask` is what the `fill-mask` pipeline does under the hood:

In [ ]:
unmasker = pipeline("fill-mask", model=bert, tokenizer=bert_tokenizer)
for prediction in unmasker("The capital of France is [MASK]."):
    print(f"    {prediction['token_str']!r:<16} {prediction['score']:.3f}")

### Context on both sides

**1.2 Run the cell below.** In each pair, the words *before* the mask are identical.

In [ ]:
for text in [
    "I went to the [MASK] to buy some bread.",
    "I went to the [MASK] to see a doctor.",
    "The [MASK] barked at the postman all night.",
    "The [MASK] purred on my lap all night.",
]:
    print(text)
    show(fill_mask(text))

**1.3 What does this show?** What would a left-to-right model like your mini GPT (or GPT-2) predict at the position of the mask, and why can BERT do better? Conversely: why is BERT not a good model for *generating* text?

---

*Answer:*

The words before the mask are the same in each pair, so a left-to-right model would have to predict the **same distribution** in both sentences — something like `store`, `shop`, `hospital`, … for the first pair, and any animal for the second. BERT sees the words *after* the mask as well (`to buy some bread` vs. `to see a doctor`, `barked` vs. `purred`), and its predictions change accordingly (bakery/store/shop vs. hospital/doctor/clinic, dog vs. cat). This is what bidirectional self-attention buys you: every token's representation is built from the whole sentence, which is why encoders are so good at *understanding* tasks.

The price is that BERT has never learned to produce text one token at a time. It was trained to fill in about 15% of hidden tokens in an otherwise complete text, with the rest of the sentence visible. To generate, you would have to start with only masks and fill them in some order, and each prediction would be made without the context the model was trained with. A model trained with a causal mask, on the other hand, does at training time exactly what it does at generation time.

---

### What else did BERT learn?

BERT learned everything it knows from its pretraining text (English Wikipedia and the BooksCorpus collection of books). That includes facts about the world — and also the patterns of how people are written about.

**1.4 Run the cell below**, which compares the probabilities of *he* and *she* at the mask. Add a few sentences of your own.

In [ ]:
he_id, she_id = bert_tokenizer.convert_tokens_to_ids(["he", "she"])

for job in ["nurse", "engineer", "secretary", "doctor", "teacher", "pilot"]:
    probs = mask_distribution(f"The {job} said that [MASK] would be late.")
    print(f"{job:<12} he: {probs[he_id]:.3f}   she: {probs[she_id]:.3f}")

**1.5 Describe the pattern.** Where does it come from — is it a bug in the model? Why does it matter if you fine-tune BERT for a real application, for example to rank job applications?

---

*Answer:*

For most jobs one of the two pronouns is clearly preferred, and the preference follows the usual stereotypes: *she* for nurse and secretary, *he* for engineer, pilot and (less strongly) doctor. Nothing in the sentence says anything about the person's gender.

It is not a bug in the code — it is the pretraining objective doing exactly what it should. The model learned to predict the words that are likely *in its training text*, and in Wikipedia and in books nurses are more often referred to as *she* and engineers as *he*. A language model reproduces the statistics of its data, including the social ones.

When the model is fine-tuned, these associations do not disappear: the fine-tuned model starts from the same representations, in which "nurse" is closer to female words and "engineer" to male words. A CV-screening model built on top of BERT can therefore rank applicants differently by gender — through a name, a pronoun or a women's football club — even if gender was never a feature, and even if the fine-tuning data was balanced. Such effects have to be measured explicitly before a model is used for decisions about people.

---

### Contextual word vectors

In week 3 you worked with **static** word embeddings: word2vec gives the word *bank* one vector, whatever it means in a sentence. BERT's output vectors are **contextual**: every occurrence of a token gets its own vector, computed from the whole sentence.

**1.6 Complete `word_vector`**, which returns BERT's last-layer vector for a word in a sentence.

Hints:
- With `output_hidden_states=True` the model also returns `.hidden_states`: a tuple with the output of the embedding layer and of each of the 12 layers, each of shape `(1, seq_len, 768)`. The last element is the output of the last layer.
- The word must be a single token in BERT's vocabulary; find its position with `bert_tokenizer.convert_tokens_to_ids(word)`.

In [ ]:
def word_vector(sentence, word):
    # BERT's last-layer vector of `word` (a single token) in `sentence`, shape (768,)
    inputs = bert_tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        hidden = bert(**inputs, output_hidden_states=True).hidden_states[-1][0]
    position = (inputs["input_ids"][0] == bert_tokenizer.convert_tokens_to_ids(word)).nonzero()[0].item()
    return hidden[position]

In [ ]:
# ✅ Check your word_vector
v1 = word_vector("He sat on the bank of the river.", "bank")
v2 = word_vector("She works at the bank.", "bank")
assert v1.shape == (768,), f"expected a vector of size 768, got shape {tuple(v1.shape)}"
assert not torch.allclose(v1, v2), "the same word in two sentences should get two different vectors"
assert torch.allclose(v1, word_vector("He sat on the bank of the river.", "bank")), "the same sentence should give the same vector"
print("Looks good ✅")

**1.7 Run the cell below.** It compares the vectors of *bank* in four sentences with the cosine similarity.

In [ ]:
sentences = [
    "He sat on the bank of the river and watched the water.",
    "The fishermen pulled their boat onto the muddy bank.",
    "She deposited the money at the bank this morning.",
    "The bank raised its interest rates again.",
]
vectors = torch.stack([word_vector(s, "bank") for s in sentences])
similarity = F.cosine_similarity(vectors[:, None], vectors[None, :], dim=-1)

for i, s in enumerate(sentences):
    print(f"{i}: {s}")
print("\n     " + "".join(f"{j:>7}" for j in range(len(sentences))))
for i in range(len(sentences)):
    print(f"{i:>3}  " + "".join(f"{similarity[i, j]:>7.2f}" for j in range(len(sentences))))

**1.8 Describe the similarity matrix.** What would the matrix look like with word2vec? Why are the vectors of the same *sense* not identical, and why are vectors of *different* senses still fairly similar?

---

*Answer:*

The two river banks (0 and 1) are more similar to each other than to either of the financial banks, and the two financial banks (2 and 3) are more similar to each other than to the river banks: the vectors group by **meaning**, not just by spelling. BERT has disambiguated the word from its context, without ever being told that *bank* has two senses.

With word2vec every entry would be **1.0**: there is one vector for *bank*, whatever the sentence.

The vectors of the same sense are not identical because each vector encodes the whole context — the other words, the position, the syntax of the sentence — not only the sense. And vectors of different senses are still fairly similar (BERT's vectors are generally quite similar to each other, and in all four sentences the token is a noun spelled *bank*), so it is the *relative* similarities that carry the information.

---

## 2. GPT-2: your mini GPT, grown up

[GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) (Radford et al., 2019) is a decoder-only Transformer trained to predict the next token on 40 GB of web text. We use the smallest version, GPT-2 *small*. `AutoModelForCausalLM` loads it with its language-modelling head.

In [ ]:
GPT2_NAME = "openai-community/gpt2"

gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_NAME)
gpt2 = AutoModelForCausalLM.from_pretrained(GPT2_NAME).to(device).eval()

print(gpt2)

Compare the printout with your `GPT` class: `wte` and `wpe` are the token and position embeddings, `h` is the list of blocks, each with `ln_1`, `attn`, `ln_2` and `mlp`, followed by `ln_f` and the `lm_head`. `c_attn` is your `qkv` layer — note its output size. (`Conv1D` is simply a linear layer that stores its weight matrix transposed, a leftover from the original TensorFlow code.)

**2.1 Count the parameters of GPT-2, and check whether its head is tied to the token embedding.**

Hint: `gpt2.get_input_embeddings()` and `gpt2.get_output_embeddings()` return the two layers. Two tensors share the same memory if their `.data_ptr()` is the same.

In [ ]:
n_params = sum(p.numel() for p in gpt2.parameters())
tied = gpt2.get_output_embeddings().weight.data_ptr() == gpt2.get_input_embeddings().weight.data_ptr()

print(f"{n_params:,} parameters, tied: {tied}")

In [ ]:
# ✅ Check your numbers
assert 120e6 < n_params < 130e6, "GPT-2 small has about 124 million parameters"
assert tied is True, "GPT-2 ties its output layer to the token embedding"
print("Looks good ✅")

### Subword tokens

Instead of characters, GPT-2 reads **byte-level BPE** tokens: frequent words are a single token, rare words are split into pieces. `Ġ` marks a token that starts with a space. You will learn how such a vocabulary is built later in the course.

In [ ]:
text = "Shakespeare wrote: To be, or not to be. Pretraining is unbelievably expensive!"
tokens = gpt2_tokenizer.tokenize(text)
print(f"{len(text)} characters -> {len(tokens)} tokens")
print(tokens)

**2.2 Your mini GPT and GPT-2 small.** Using the printout and the cells above, compare the two models: number of layers, embedding dimension, context length, vocabulary, and how much text fits into the context. What is the same in the two architectures?

---

*Answer:*

| | mini GPT | GPT-2 small |
|---|---|---|
| layers | 4 | 12 |
| embedding dimension | 128 | 768 (`c_attn`: 768 → 2,304 = 3 × 768, the same fused q/k/v as `qkv`) |
| heads | 4 | 12 |
| context | 64 tokens | 1,024 tokens (`wpe` has 1,024 rows) |
| vocabulary | 65 characters | 50,257 subword tokens |
| parameters | 0.8 M | 124 M |

With characters, 64 tokens are about one line of text. A GPT-2 token covers about four characters of English on average (compare the number of characters and tokens in the example above), so its 1,024 tokens hold several pages — 16 times as many tokens, each covering about four times as much text.

The architecture is otherwise **the same**: token plus learned position embeddings, pre-norm blocks with causal self-attention (fused q/k/v projection) and a GELU MLP with a 4× hidden layer, a final layer norm, and a head tied to the token embedding.

---

### The next token

**2.3 Complete `next_token_distribution`**, which returns GPT-2's most likely next tokens after a prompt.

Hint: the model returns logits for every position of the prompt. Which position predicts the token *after* the prompt?

In [ ]:
def next_token_distribution(prompt, k=10):
    # The k most likely tokens after the prompt, as a list of (token, probability)
    inputs = gpt2_tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = gpt2(**inputs).logits
    probs = logits[0, -1].softmax(dim=-1)
    top = torch.topk(probs, k)
    return [(gpt2_tokenizer.decode([i]), p) for i, p in zip(top.indices.tolist(), top.values.tolist())]


show(next_token_distribution("The capital of France is"))

In [ ]:
# ✅ Check your next_token_distribution
top = next_token_distribution("The capital of France is")
assert len(top) == 10 and all(a[1] >= b[1] for a, b in zip(top, top[1:])), "expected the 10 most likely tokens, sorted by probability"
assert any(token.strip() == "Paris" for token, _ in top), "' Paris' should be among the 10 most likely next tokens — are you using the last position?"
print("Looks good ✅")

Unlike your Shakespeare model, GPT-2 knows that Paris is the capital of France — but it is not necessarily its first choice. Try a few prompts of your own:

In [ ]:
for prompt in [
    "The Eiffel Tower is located in the city of",
    "Aarhus is the second largest city in",
    "Q: What is 12 plus 7?\nA:",
]:
    print(repr(prompt))
    show(next_token_distribution(prompt, k=5))

### Perplexity

A language model assigns a probability to every text. The **perplexity** (week 3) turns it into a number: the exponential of the average negative log-probability of the tokens. The lower, the less surprised the model is.

If you pass `labels` to the model, it computes the loss itself — the same next-token cross-entropy you computed in your `GPT.forward`. You pass the input ids as the labels; the model shifts them by one position internally.

**2.4 Complete `perplexity`.**

In [ ]:
def perplexity(text):
    inputs = gpt2_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        loss = gpt2(**inputs, labels=inputs["input_ids"]).loss
    return math.exp(loss.item())


print(perplexity("The cat sat on the mat."))

In [ ]:
# ✅ Check your perplexity against a computation by hand
text = "The cat sat on the mat."
inputs = gpt2_tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
    logits = gpt2(**inputs).logits
by_hand = math.exp(F.cross_entropy(logits[0, :-1], inputs["input_ids"][0, 1:]).item())  # position t predicts token t + 1
assert abs(perplexity(text) - by_hand) < 1e-3 * by_hand, f"expected {by_hand:.2f}, got {perplexity(text):.2f}"
print("Looks good ✅")

**2.5 Run the cell below**, and add texts of your own.

In [ ]:
for text in [
    "The cat sat on the mat.",
    "The mat sat on the cat.",
    "mat the on sat cat The.",
    "Colorless green ideas sleep furiously.",
    "To be, or not to be, that is the question.",
    "Katten sad på måtten.",
    "for i in range(10):\n    print(i)",
]:
    print(f"{perplexity(text):>10.1f}   {text!r}")

**2.6 Explain the differences.** Which texts surprise GPT-2 the most, and why? Can you compare these numbers with the validation loss of your Shakespeare model from notebook 0?

---

*Answer:*

- The normal sentence gets a low perplexity. *The mat sat on the cat* is grammatical but less likely (mats do not sit), so its perplexity is higher. The **shuffled** words are by far the most surprising English text: every word is a plausible English word, but the order is not.
- *Colorless green ideas sleep furiously* (Chomsky's example of a grammatical but meaningless sentence) is also very surprising: GPT-2 has learned not only syntax but also which words go together.
- The **Hamlet** quote gets a very low perplexity: it occurs many times on the web, and the model has practically memorised it.
- **Danish** is surprising: GPT-2 was trained on (mostly) English web text, and its BPE vocabulary splits Danish words into many small, rare pieces. **Code** is often surprisingly predictable, because there is a lot of it on the web.

Comparing with notebook 0 is **not fair**: the perplexity is measured *per token*, and the tokens are different. The Shakespeare model's loss is per character, while a GPT-2 token covers about four characters, so predicting a GPT-2 token is a much harder task. To compare models with different tokenizers you have to normalise by a unit that does not depend on the tokenizer, for example bits per character or per byte.

---

### Generating text

You implemented sampling with temperature and top-k yourself. Hugging Face models have a `generate` method that does this (and much more). Its most important arguments:

- `max_new_tokens`: how many tokens to generate,
- `do_sample`: `False` for greedy decoding, `True` for sampling,
- `temperature` and `top_k`: as in your `generate`,
- `top_p` (*nucleus sampling*): keep the smallest set of tokens whose probabilities add up to `top_p`, so the number of candidates adapts to how certain the model is.

GPT-2 has no padding token, so we pass `pad_token_id=gpt2_tokenizer.eos_token_id` to avoid a warning.

**2.7 Generate a continuation of the prompt with greedy decoding and with nucleus sampling.**

In [ ]:
prompt = "In a shocking finding, scientists discovered a herd of unicorns living in a remote valley in the Andes."
inputs = gpt2_tokenizer(prompt, return_tensors="pt").to(device)

torch.manual_seed(0)
greedy = gpt2.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=gpt2_tokenizer.eos_token_id)
sampled = gpt2.generate(**inputs, max_new_tokens=80, do_sample=True, temperature=0.8, top_p=0.9, pad_token_id=gpt2_tokenizer.eos_token_id)

print("===== greedy =====")
print(gpt2_tokenizer.decode(greedy[0], skip_special_tokens=True))
print("\n===== sampled =====")
print(gpt2_tokenizer.decode(sampled[0], skip_special_tokens=True))

In [ ]:
# ✅ Check your generations
prompt_length = inputs["input_ids"].size(1)
assert prompt_length < greedy.shape[1] <= prompt_length + 80 and prompt_length < sampled.shape[1] <= prompt_length + 80, "generate up to 80 new tokens after the prompt"
again = gpt2.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=gpt2_tokenizer.eos_token_id)
assert torch.equal(greedy, again), "greedy decoding should be deterministic — did you set do_sample=False?"
print("Looks good ✅")

Now try the model as an assistant:

In [ ]:
torch.manual_seed(0)
prompt = "Question: What is the capital of Denmark?\nAnswer:"
inputs = gpt2_tokenizer(prompt, return_tensors="pt").to(device)
output = gpt2.generate(**inputs, max_new_tokens=40, do_sample=True, top_p=0.9, pad_token_id=gpt2_tokenizer.eos_token_id)
print(gpt2_tokenizer.decode(output[0], skip_special_tokens=True))

**2.8 Compare the outputs.** What typically goes wrong with greedy decoding? How does the sampled text compare? And why does GPT-2 not behave like a chat assistant, even though it was trained on far more text than your Shakespeare model?

---

*Answer:*

**Greedy decoding** usually gets stuck in a **loop**: after a sentence or two it repeats the same phrase or sentence over and over. Each step picks the single most likely token, and once a phrase has occurred, repeating it is very likely — so the model keeps doing it. The **sampled** text is more varied and reads more like human writing, but it drifts: it invents names, quotes and details, and sometimes contradicts itself.

GPT-2 is a **pretrained** model and nothing else. It has learned to continue web text, and in web text a question is not necessarily followed by a correct, helpful answer: it may be followed by another question (a quiz or a forum thread), a wrong answer, or a change of subject. The model may produce *Copenhagen*, but it is not trying to answer — it continues the document. Chat assistants are pretrained the same way and then **fine-tuned** on instruction–response examples and with human feedback, so that "continue the text" turns into "answer the user". (They are also orders of magnitude larger, which is what makes the answers more often correct.)

---

## 3. T5: one model for every task

[T5](https://arxiv.org/abs/1910.10683) (Raffel et al., 2020), the *Text-to-Text Transfer Transformer*, is an **encoder–decoder** — the architecture you built last week. It was pretrained with **span corruption**: random spans of the input are replaced by *sentinel* tokens `<extra_id_0>`, `<extra_id_1>`, …, and the decoder has to write out the missing spans, each one after its sentinel:

```
original   Thank you for inviting me to your party last week.
input      Thank you <extra_id_0> me to your party <extra_id_1> week.
target     <extra_id_0> for inviting <extra_id_1> last <extra_id_2>
```

We use T5-*small* (60 M parameters).

In [ ]:
T5_NAME = "google-t5/t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(T5_NAME)
t5 = AutoModelForSeq2SeqLM.from_pretrained(T5_NAME).to(device).eval()

print(f"{sum(p.numel() for p in t5.parameters()):,} parameters")

**3.1 Complete `t5_generate`**, which runs T5 on a text and returns its output as a string.

Hints:
- `t5.generate(**inputs, max_new_tokens=...)` returns a tensor of token ids of shape `(1, output_length)`. By default it uses greedy decoding.
- `t5_tokenizer.decode(ids, skip_special_tokens=...)`: the sentinels are special tokens, so to see them you have to keep the special tokens.

In [ ]:
def t5_generate(text, max_new_tokens=40, skip_special_tokens=True):
    inputs = t5_tokenizer(text, return_tensors="pt").to(device)
    outputs = t5.generate(**inputs, max_new_tokens=max_new_tokens)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=skip_special_tokens)


print(t5_generate("Thank you <extra_id_0> me to your party <extra_id_1> week.", skip_special_tokens=False))

In [ ]:
# ✅ Check your t5_generate
corrupted = t5_generate("Thank you <extra_id_0> me to your party <extra_id_1> week.", skip_special_tokens=False)
assert isinstance(corrupted, str), "t5_generate should return a string"
assert "<extra_id_0>" in corrupted, "the output of span corruption should contain the sentinel <extra_id_0> — keep the special tokens"
translation = t5_generate("translate English to German: The house is wonderful.")
print(translation)
assert "Haus" in translation and "<" not in translation, "expected a German translation without special tokens"
print("Looks good ✅")

### Text to text

The idea of T5 is that **every task is a text-to-text task**: the input is a text that starts with a *task prefix*, the output is a text. Translation outputs the translation; classification outputs the name of the class; even a regression task outputs a number, written as text. The released T5 checkpoints were trained on a mixture of the span-corruption objective and such supervised tasks, so they understand a few prefixes out of the box.

**3.2 Run the cell below.** Then change the inputs: other sentences, other prefixes, a typo in a prefix, or a language that T5 was not trained on (it knows German, French and Romanian).

In [ ]:
article = (
    "summarize: The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
    "It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889. "
    "Locally nicknamed 'La dame de fer', it was constructed as the centerpiece of the 1889 World's Fair, "
    "and to crown the centennial anniversary of the French Revolution. Although initially criticised by some of "
    "France's leading artists and intellectuals for its design, it has since become a global cultural icon of France "
    "and one of the most recognisable structures in the world."
)

for text in [
    "translate English to German: The house is wonderful.",
    "translate English to French: Pretraining a language model takes a lot of compute.",
    "translate English to Danish: The house is wonderful.",
    article,
    "cola sentence: The books is on the table.",
    "cola sentence: The books are on the table.",
    "sst2 sentence: This movie was a complete waste of time.",
    "stsb sentence1: A man is playing a guitar. sentence2: A person plays an instrument.",
]:
    print(f"{text[:80]!r}\n    -> {t5_generate(text, max_new_tokens=60)!r}\n")

**3.3 How does T5 solve classification and regression tasks?** What happened with the Danish prefix (or with your typos), and what does that tell you about what the prefixes are?

---

*Answer:*

T5 **writes the answer as text**. For CoLA (is the sentence grammatical?) it writes `acceptable` or `unacceptable`, for SST-2 `positive` or `negative`, and for STS-B (how similar are two sentences, from 0 to 5) a number such as `3.8`, as a string. There are no task-specific output layers: one model, one loss (the cross-entropy of the output tokens), one decoding procedure for every task. For a classifier you only have to check which label string comes out.

The prefixes are not commands that T5 *understands*; they are simply text that was always in front of the input for a given task during training, and the model has learned what kind of output follows it. With a prefix it has never seen — `translate English to Danish:`, or a typo — it does not fail with an error. It does *something*: often it produces German or French, or copies the English sentence. A model trained on a fixed set of prefixes cannot follow new instructions; that ability only came with much larger models trained on many instructions.

---

**3.4 Which family would you choose?** For each task, choose an encoder-only, a decoder-only or an encoder–decoder model, and explain briefly:
1. classify customer reviews as positive or negative,
2. a system that writes short stories from a first sentence,
3. translate product descriptions from English to Danish,
4. find the 10 most similar documents in a large collection.

---

*Answer:*

1. **Encoder-only (BERT).** The whole review is available at once, and bidirectional context gives the best representations for classification: fine-tune BERT with a small classification head on top of the `[CLS]` vector. It is small and cheap to run.
2. **Decoder-only (GPT).** Writing text left to right is exactly what a causal language model is pretrained to do.
3. **Encoder–decoder (T5, BART, or a dedicated translation model like last week's `opus-mt`).** The encoder reads the full source sentence in both directions, and the decoder writes the translation while attending to it through cross-attention. (Large decoder-only models translate well too, if you have one.)
4. **Encoder-only.** Turn every document into a vector with an encoder (in practice a BERT-like model fine-tuned to produce sentence embeddings), and compare the vectors with the cosine similarity, as in 1.7. The documents only have to be encoded once, and there is nothing to generate.

---

## 4. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 4.1. BART's hybrid design

[BART](https://arxiv.org/abs/1910.13461) combines which two ideas?

A. Bidirectional encoder + autoregressive decoder<br>
B. Masked LM + next sentence prediction<br>
C. LSTM + CNN<br>
D. Word2Vec + Transformers<br>

**Answer:** **A**

BART is an encoder–decoder: a BERT-like bidirectional encoder reads the corrupted input, and a GPT-like left-to-right decoder writes the original text. Masked LM and next sentence prediction (B) are BERT's two pretraining objectives.

---

### 4.2. BART vs BERT

Which of the following is TRUE about BART compared to BERT?

A. BART cannot handle generative tasks<br>
B. BART performs worse on classification tasks<br>
C. BART can do both understanding and generation<br>
D. BART uses only unidirectional encoding<br>

**Answer:** **C**

Thanks to its decoder, BART can generate text (summarisation, translation, question answering with free-form answers), which BERT cannot do well (1.3); and its bidirectional encoder makes it about as good as comparable BERT-style models on classification benchmarks such as GLUE. A, B and D are therefore wrong.

---

### 4.3. BART architecture

What architecture does BART use?

A. Encoder-only<br>
B. Decoder-only<br>
C. Encoder–decoder with cross-attention<br>
D. RNN-based encoder<br>

**Answer:** **C**

The standard Transformer encoder–decoder — the same structure as T5 and as the model you built last week: the decoder attends to the encoder output through cross-attention.

---

### 4.4. BART's advantage

Why does BART outperform BERT on some tasks?

A. It has fewer parameters<br>
B. It was trained on significantly more data<br>
C. It does not use positional embeddings<br>
D. It uses static word embeddings<br>

**Answer:** **B**

BART was pretrained on about 160 GB of text — the same data as RoBERTa — while BERT used about 16 GB (Wikipedia and BooksCorpus). As your mini GPT showed, the amount of pretraining data matters a lot. (Its denoising objective and its decoder help as well, especially for generation.) A is wrong (BART-large is larger than BERT-large), C is wrong (it uses learned position embeddings), and D is wrong (all of these models produce contextual representations).

---

### 4.5. BART corruption strategies

Which of the following corruption strategies can BART use during pretraining?

A. Masking tokens randomly<br>
B. Permuting sentences<br>
C. Replacing tokens with random ones<br>
D. All of the above<br>

**Answer:** **D**

BART is a *denoising autoencoder*: the input can be corrupted in any way, because the decoder always has to reproduce the complete original text. The paper tried token masking, token deletion, text infilling (a whole span replaced by a single mask — the variant that worked best), sentence permutation and document rotation. (Strictly speaking, random replacement is part of BERT's masking recipe rather than one of BART's named strategies, but the framework allows any corruption — the intended answer is D.)

---

### 4.6. T5's name

What does T5 stand for?

A. Text-to-Text Transfer Transformer<br>
B. Transformer for 5 tasks<br>
C. Transferable Transformer Training Technique<br>
D. Text Transformer for Translation<br>

**Answer:** **A**

**T**ext-**T**o-**T**ext **T**ransfer **T**ransformer — five T's.

---

### 4.7. T5's philosophy

What is the key design philosophy of T5?

A. Every NLP task can be framed as a text-to-text problem<br>
B. Using unidirectional LSTMs<br>
C. Using static embeddings<br>
D. Relying only on classification tasks<br>

**Answer:** **A**

Input text in, output text out, for every task — even classification and regression, as you saw in 3.2. This allows one model, one loss and one decoding procedure for all tasks.

---

### 4.8. T5's pretraining objective

Which pretraining objective did T5 use?

A. Masked language modeling (like BERT)<br>
B. Next sentence prediction<br>
C. Denoising span corruption (span masking)<br>
D. Causal LM (like GPT)<br>

**Answer:** **C**

Spans of the input are replaced by sentinel tokens, and the decoder writes out the missing spans (3.1). Unlike BERT's masked LM (A), a whole span is replaced by a single sentinel, and the target contains only the missing parts, not the whole text — which makes pretraining cheaper.

---

### 4.9. T5's architecture

Which type of Transformer architecture does T5 employ?

A. Encoder-only<br>
B. Decoder-only<br>
C. Encoder–decoder<br>
D. Hybrid LSTM-Transformer<br>

**Answer:** **C**

T5 uses the original encoder–decoder Transformer; the paper compared it with encoder-only and decoder-only variants, and the encoder–decoder worked best for its text-to-text setting.

---

### 4.10. T5's limitations

Which of the following is NOT a task T5 can handle natively in its framework?

A. Machine translation<br>
B. Summarization<br>
C. Image classification<br>
D. Question answering<br>

**Answer:** **C**

T5's inputs and outputs are text. Translation, summarisation and question answering are text-to-text tasks; an image is not text, so T5 cannot read it without an additional vision component.